In [ ]:
from pyspark.sql.functions import (
    length, size, split, regexp_replace, col
)

# 1 char count
spark_df = spark_df.withColumn("char_count", length(col("text")))

# 2 word count
spark_df = spark_df.withColumn("word_count", size(split(col("text"), " ")))

# 3 sentence count
spark_df = spark_df.withColumn(
    "sentence_count",
    length(regexp_replace(col("text"), "[^.!?]", ""))
)

# 4 punctuation count
spark_df = spark_df.withColumn(
    "punctuation_count",
    length(regexp_replace(col("text"), "[^\\p{Punct}]", ""))
)

# 5 digit count
spark_df = spark_df.withColumn(
    "digit_count",
    length(regexp_replace(col("text"), "[^0-9]", ""))
)

# 6 uppercase count
spark_df = spark_df.withColumn(
    "uppercase_count",
    length(regexp_replace(col("text"), "[^A-Z]", ""))
)

# 7 uppercase ratio
spark_df = spark_df.withColumn(
    "uppercase_ratio",
    col("uppercase_count") / col("char_count")
)

# 8 digit ratio
spark_df = spark_df.withColumn(
    "digit_ratio",
    col("digit_count") / col("char_count")
)

# 9 newline count
spark_df = spark_df.withColumn(
    "newline_count",
    length(regexp_replace(col("text"), "[^\\n]", ""))
)

# 10 average word length
spark_df = spark_df.withColumn(
    "avg_word_length",
    col("char_count") / col("word_count")
)

spark_df.show(5)

In [ ]:
# Add unique document ID
spark_df = spark_df.withColumn("doc_id", monotonically_increasing_id())

# UDF for unique word count
def count_unique_words(text):
    if not text:
        return 0
    words = text.lower().split()
    return len(set(words))

unique_words_udf = udf(count_unique_words, IntegerType())
spark_df = spark_df.withColumn("unique_word_count", unique_words_udf(col("text")))

# 11. Lexical diversity (unique words / total words)
spark_df = spark_df.withColumn(
    "lexical_diversity",
    when(col("word_count") > 0, col("unique_word_count") / col("word_count")).otherwise(0.0)
)

# 12. Average sentence length (words per sentence)
spark_df = spark_df.withColumn(
    "avg_sentence_length",
    when(col("sentence_count") > 0, col("word_count") / col("sentence_count")).otherwise(0.0)
)

# 13. Punctuation ratio
spark_df = spark_df.withColumn(
    "punctuation_ratio",
    when(col("char_count") > 0, col("punctuation_count") / col("char_count")).otherwise(0.0)
)

# 14. Space count and ratio
spark_df = spark_df.withColumn(
    "space_count",
    length(regexp_replace(col("text"), "[^ ]", ""))
)
spark_df = spark_df.withColumn(
    "space_ratio",
    when(col("char_count") > 0, col("space_count") / col("char_count")).otherwise(0.0)
)

# 15. Special character count
spark_df = spark_df.withColumn(
    "special_char_count",
    length(regexp_replace(col("text"), "[a-zA-Z0-9\\s]", ""))
)

# 16. Alphabetic characters
spark_df = spark_df.withColumn(
    "alphabetic_count",
    length(regexp_replace(col("text"), "[^a-zA-Z]", ""))
)
spark_df = spark_df.withColumn(
    "alphabetic_ratio",
    when(col("char_count") > 0, col("alphabetic_count") / col("char_count")).otherwise(0.0)
)

print(f"Advanced features added. Total columns: {len(spark_df.columns)}")
spark_df.printSchema()